In [0]:
from src.transformations import transform_hdb_data
from pyspark.sql.functions import col, sum

In [0]:
df_bronze = spark.table(
    "workspace.hdb_pyspark.bronze_hdb_resale"
    )

In [0]:
df_bronze.count()

In [0]:
df_bronze.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
]).show()

In [0]:
df_data = df_bronze.drop("ingested_at")

dup_count = (
    df_data.count() - 
    df_data.dropDuplicates().count()
)

print(dup_count)

In [0]:
dup_groups.select(
    "month",
    "town",
    "block",
    "street_name",
    "resale_price",
    "count"
).show(10, truncate=False)

In [0]:
df_bronze.select(
    "floor_area_sqm",
    "lease_commence_date",
    "resale_price"
).describe().show()

In [0]:
df_silver = transform_hdb_data(df_bronze)

In [0]:
df_silver.select(
    "month",
    "transaction_date",
    "transaction_year",
    "transaction_month",
    "resale_price",
    "floor_area_sqm",
    "price_per_sqm",
    "flat_age",
    "remaining_lease",
    "remaining_lease_years",
    "remaining_lease_months"
).show(10, truncate=False)

In [0]:
#validate transformed data
df_silver.selectExpr(
    "sum(CASE WHEN transaction_date IS NULL THEN 1 ELSE 0 END) AS invalid_dates",
    "sum(CASE WHEN price_per_sqm <= 0 THEN 1 ELSE 0 END) AS invalid_price_per_sqm",
    "sum(CASE WHEN flat_age < 0 THEN 1 ELSE 0 END) AS invalid_flat_age",
    "sum(CASE WHEN remaining_lease_years IS NULL THEN 1 ELSE 0 END) AS invalid_remaining_lease"
).show()

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.hdb_pyspark.silver_hdb_resale")
    )

In [0]:
#verify
spark.table(
    "workspace.hdb_pyspark.silver_hdb_resale"
).show(5, truncate=False)

In [0]:
df_silver.selectExpr(
    "sum(CASE WHEN transaction_date IS NULL THEN 1 ELSE 0 END) AS invalid_dates",
    "sum(CASE WHEN price_per_sqm <= 0 THEN 1 ELSE 0 END) AS invalid_price_per_sqm",
    "sum(CASE WHEN flat_age < 0 THEN 1 ELSE 0 END) AS invalid_flat_age",
    "sum(CASE WHEN remaining_lease_years IS NULL THEN 1 ELSE 0 END) AS invalid_remaining_lease"
).show()

In [0]:
bronze_count = spark.table(
    "workspace.hdb_pyspark.bronze_hdb_resale"
).count()

silver_count = spark.table(
    "workspace.hdb_pyspark.silver_hdb_resale"
).count()

print("Bronze:", bronze_count)
print("Silver:", silver_count)